In [41]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
from abc import ABC, abstractmethod
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report



Своя нейронная сеть для классификации

In [42]:
class Layer(ABC):
    @abstractmethod
    def forward(self, X):
        pass
    
    @abstractmethod
    def backward(self, grad_output, learning_rate):
        pass

class Dense(Layer):
    def __init__(self, input_size, output_size, activation=None):
        self.input_size = input_size
        self.output_size = output_size
        self.activation = activation
        
        self.W = np.random.randn(input_size, output_size) * np.sqrt(2 / input_size)
        self.b = np.zeros((1, output_size))
        
        self.X = None
        self.Z = None
    
    def forward(self, X):
        self.X = X
        self.Z = X @ self.W + self.b
        
        if self.activation:
            return self.activation.forward(self.Z)
        return self.Z
    
    def backward(self, grad_output, learning_rate):
        if self.activation:
            grad = self.activation.backward(self.Z, grad_output)
        else:
            grad = grad_output
        
        if grad.ndim == 1:
            grad = grad.reshape(-1, 1)
        
        m = self.X.shape[0]
        dW = (self.X.T @ grad) / m
        db = np.mean(grad, axis=0, keepdims=True)
        
        grad_input = grad @ self.W.T
        
        self.W -= learning_rate * dW
        self.b -= learning_rate * db
        
        return grad_input

class Activation(ABC):
    @abstractmethod
    def forward(self, Z):
        pass
    
    @abstractmethod
    def backward(self, Z, grad_output):
        pass

class ReLU(Activation):
    def forward(self, Z):
        return np.maximum(0, Z)
    
    def backward(self, Z, grad_output):
        return grad_output * (Z > 0).astype(float)

class Sigmoid(Activation):
    def forward(self, Z):
        return 1 / (1 + np.exp(-np.clip(Z, -500, 500)))
    
    def backward(self, Z, grad_output):
        A = self.forward(Z)
        return grad_output * A * (1 - A)

class NeuralNetwork:
    def __init__(self):
        self.layers = []
    
    def add(self, layer):
        self.layers.append(layer)
        return self
    
    def forward(self, X):
        for layer in self.layers:
            X = layer.forward(X)
        return X
    
    def backward(self, grad_output, learning_rate):
        for layer in reversed(self.layers):
            grad_output = layer.backward(grad_output, learning_rate)
        return grad_output
    
    def fit(self, X, y, epochs=1000, learning_rate=0.01, batch_size=None, verbose=True):
        if y.ndim == 1:
            y = y.reshape(-1, 1)
        
        n_samples = X.shape[0]
        
        for epoch in range(epochs):
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            
            if batch_size:
                for i in range(0, n_samples, batch_size):
                    X_batch = X_shuffled[i:i+batch_size]
                    y_batch = y_shuffled[i:i+batch_size]
                    self._train_batch(X_batch, y_batch, learning_rate)
            else:
                self._train_batch(X_shuffled, y_shuffled, learning_rate)
            
            if verbose and epoch % 100 == 0:
                y_pred = self.forward(X)
                loss = self._compute_loss(y, y_pred)
                print(f"Epoch {epoch:4d} | Loss: {loss:.6f}")
            elif epoch == epochs - 1:
                y_pred = self.forward(X)
                loss = self._compute_loss(y, y_pred)
                if verbose:
                    print(f"Epoch {epoch:4d} | Loss: {loss:.6f}")
        
        return self
    
    def _train_batch(self, X_batch, y_batch, learning_rate):
        y_pred = self.forward(X_batch)
        grad_output = self._loss_gradient(y_batch, y_pred)
        self.backward(grad_output, learning_rate)
    
    def _compute_loss(self, y_true, y_pred):
        eps = 1e-8
        return -np.mean(y_true * np.log(y_pred + eps) + (1 - y_true) * np.log(1 - y_pred + eps))
    
    def _loss_gradient(self, y_true, y_pred):
        return y_pred - y_true
    
    def predict(self, X):
        return self.forward(X)
    
    def predict_class(self, X, threshold=0.5):
        return (self.predict(X) >= threshold).astype(int)
    
    def score(self, X, y):
        if y.ndim == 1:
            y = y.reshape(-1, 1)
        y_pred = self.predict_class(X)
        return np.mean(y_pred == y)

In [43]:

data = pd.read_csv('train_data.csv')
X = data.drop(['PassengerId', 'Survived'], axis=1).values
y = data['Survived'].values

print(f"Признаков: {X.shape[1]}, Объектов: {X.shape[0]}")

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = NeuralNetwork()

model.add(Dense(X.shape[1], 32, activation=ReLU()))
model.add(Dense(32, 64, activation=ReLU()))
model.add(Dense(64, 32, activation=ReLU()))
model.add(Dense(32, 16, activation=ReLU()))
model.add(Dense(16, 1, activation=Sigmoid()))

model.fit(X_train, y_train, epochs=1000, learning_rate=0.01, batch_size=16, verbose=True)

print(f"Точность: {model.score(X_test, y_test):.4f}")

Признаков: 15, Объектов: 792
Epoch    0 | Loss: 0.672397
Epoch  100 | Loss: 0.395975
Epoch  200 | Loss: 0.369340
Epoch  300 | Loss: 0.354314
Epoch  400 | Loss: 0.342030
Epoch  500 | Loss: 0.329363
Epoch  600 | Loss: 0.317822
Epoch  700 | Loss: 0.304843
Epoch  800 | Loss: 0.291702
Epoch  900 | Loss: 0.279817
Epoch  999 | Loss: 0.271625
Точность: 0.8050


Тенсер Флоу Классификация

In [ ]:
df1 = pd.read_csv("customer_data.csv")
df2 = pd.read_csv("payment_data.csv")
data = pd.merge(df1, df2, on='id', how='inner')
print(data.columns.tolist())
data = data.dropna(axis=1, how='all')  
data.fillna(data.mean(numeric_only=True), inplace=True)


X = data.drop(['label', 'id', 'update_date', 'report_date','OVD_sum','new_balance','highest_balance' ], axis=1).values

y = data['label'].values

scaler_X = StandardScaler()
X = scaler_X.fit_transform(X)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify= y)

print(f"Признаков: {X.shape[1]}, Объектов: {X.shape[0]}")

model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(X.shape[1],)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)


history = model.fit(
    X_train, y_train,
    epochs=120 ,
    batch_size=32,
    validation_split=0.2,
    class_weight={0: 0.6, 1: 2.97},
    verbose=1
)


['label', 'id', 'fea_1', 'fea_2', 'fea_3', 'fea_4', 'fea_5', 'fea_6', 'fea_7', 'fea_8', 'fea_9', 'fea_10', 'fea_11', 'OVD_t1', 'OVD_t2', 'OVD_t3', 'OVD_sum', 'pay_normal', 'prod_code', 'prod_limit', 'update_date', 'new_balance', 'highest_balance', 'report_date']
Признаков: 17, Объектов: 8250


c:\Users\Lenovo\miniconda3\envs\jupyter_env\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/120
165/165 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5377 - loss: 0.6945 - val_accuracy: 0.4985 - val_loss: 0.6843
Epoch 2/120
165/165 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5765 - loss: 0.6737 - val_accuracy: 0.5614 - val_loss: 0.6726
Epoch 3/120
165/165 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5909 - loss: 0.6566 - val_accuracy: 0.5417 - val_loss: 0.6786
Epoch 4/120
165/165 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5919 - loss: 0.6554 - val_accuracy: 0.5038 - val_loss: 0.6933
Epoch 5/120
165/165 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5917 - loss: 0.6472 - val_accuracy: 0.5932 - val_loss: 0.6490
Epoch 6/120
165/165 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6322 - loss: 0.6320 - val_accuracy: 0.5568 - val_loss: 0.6783
Epoch 7/120
165/165 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6297 - loss: 0.6312 - val_accuracy: 0.6114 - val_loss: 0.6465
Epoch 8/120
165/165 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6458 - loss: 0.6149 - val_accu

In [45]:
X_df = data.drop(['label', 'id', 'update_date', 'report_date'], axis=1)
corr_matrix = X_df.corr()

print(model.evaluate(X_test, y_test)) 
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)
print(classification_report(y_test, y_pred))

52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8491 - loss: 0.3089 
[0.30893248319625854, 0.8490909337997437]
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
              precision    recall  f1-score   support

           0       0.97      0.84      0.90      1373
           1       0.53      0.89      0.66       277

    accuracy                           0.85      1650
   macro avg       0.75      0.86      0.78      1650
weighted avg       0.90      0.85      0.86      1650



Результаты своей нейронной сети для классификации на данных о платежах

In [47]:
df1 = pd.read_csv("customer_data.csv")
df2 = pd.read_csv("payment_data.csv")
data = pd.merge(df1, df2, on='id', how='inner')
print(data.columns.tolist())
data = data.dropna(axis=1, how='all')  
data.fillna(data.mean(numeric_only=True), inplace=True)


X = data.drop(['label', 'id', 'update_date', 'report_date','OVD_sum','new_balance','highest_balance' ], axis=1).values

y = data['label'].values

scaler_X = StandardScaler()
X = scaler_X.fit_transform(X)




X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify= y)

print(f"Признаков: {X.shape[1]}, Объектов: {X.shape[0]}")
model = NeuralNetwork()

model.add(Dense(X.shape[1], 32, activation=ReLU()))
model.add(Dense(32, 64, activation=ReLU()))
model.add(Dense(64, 32, activation=ReLU()))
model.add(Dense(32, 16, activation=ReLU()))
model.add(Dense(16, 1, activation=Sigmoid()))

model.fit(X_train, y_train, epochs=300, learning_rate=0.01, batch_size=16, verbose=True)

['label', 'id', 'fea_1', 'fea_2', 'fea_3', 'fea_4', 'fea_5', 'fea_6', 'fea_7', 'fea_8', 'fea_9', 'fea_10', 'fea_11', 'OVD_t1', 'OVD_t2', 'OVD_t3', 'OVD_sum', 'pay_normal', 'prod_code', 'prod_limit', 'update_date', 'new_balance', 'highest_balance', 'report_date']
Признаков: 17, Объектов: 8250
Epoch    0 | Loss: 0.507120
Epoch  100 | Loss: 0.319663
Epoch  200 | Loss: 0.234975
Epoch  299 | Loss: 0.200077


In [49]:
print(model.score(X_test, y_test)) 
y_pred = model.predict_class(X_test, 0.6)

print(classification_report(y_test, y_pred))

0.9145454545454546
              precision    recall  f1-score   support

           0       0.92      0.98      0.95      1373
           1       0.86      0.59      0.70       277

    accuracy                           0.92      1650
   macro avg       0.89      0.79      0.83      1650
weighted avg       0.91      0.92      0.91      1650



Нейронная сеть для регрессии

In [ ]:
class Layer(ABC):
    @abstractmethod
    def forward(self, X):
        pass
    
    @abstractmethod
    def backward(self, grad_output, learning_rate):
        pass

class Dense(Layer):
    def __init__(self, input_size, output_size, activation=None):
        self.input_size = input_size
        self.output_size = output_size
        self.activation = activation
        
        self.W = np.random.randn(input_size, output_size) * np.sqrt(2 / input_size)
        self.b = np.zeros((1, output_size))
        
        self.X = None
        self.Z = None
    
    def forward(self, X):
        self.X = X
        self.Z = X @ self.W + self.b
        
        if self.activation:
            return self.activation.forward(self.Z)
        return self.Z
    
    def backward(self, grad_output, learning_rate):
        if self.activation:
            grad = self.activation.backward(self.Z, grad_output)
        else:
            grad = grad_output
        
        if grad.ndim == 1:
            grad = grad.reshape(-1, 1)
        
        m = self.X.shape[0]
        dW = (self.X.T @ grad) / m
        db = np.mean(grad, axis=0, keepdims=True)
        
        grad_input = grad @ self.W.T
        
        self.W -= learning_rate * dW
        self.b -= learning_rate * db
        
        return grad_input

class Activation(ABC):
    @abstractmethod
    def forward(self, Z):
        pass
    
    @abstractmethod
    def backward(self, Z, grad_output):
        pass

class ReLU(Activation):
    def forward(self, Z):
        return np.maximum(0, Z)
    
    def backward(self, Z, grad_output):
        return grad_output * (Z > 0).astype(float)

class Sigmoid(Activation):
    def forward(self, Z):
        return 1 / (1 + np.exp(-np.clip(Z, -500, 500)))
    
    def backward(self, Z, grad_output):
        A = self.forward(Z)
        return grad_output * A * (1 - A)

class Linear(Activation):
    def forward(self, Z):
        return Z
    
    def backward(self, Z, grad_output):
        return grad_output

class NeuralNetworRegration:
    def __init__(self):
        self.layers = []
    
    def add(self, layer):
        self.layers.append(layer)
        return self
    
    def forward(self, X):
        for layer in self.layers:
            X = layer.forward(X)
        return X
    
    def backward(self, grad_output, learning_rate):
        for layer in reversed(self.layers):
            grad_output = layer.backward(grad_output, learning_rate)
        return grad_output
    
    def fit(self, X, y, epochs=1000, learning_rate=0.01, batch_size=None, verbose=True):
        if y.ndim == 1:
            y = y.reshape(-1, 1)
        
        n_samples = X.shape[0]
        
        for epoch in range(epochs):
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            
            if batch_size:
                for i in range(0, n_samples, batch_size):
                    X_batch = X_shuffled[i:i+batch_size]
                    y_batch = y_shuffled[i:i+batch_size]
                    self._train_batch(X_batch, y_batch, learning_rate)
            else:
                self._train_batch(X_shuffled, y_shuffled, learning_rate)
            
            if verbose and epoch % 100 == 0:
                y_pred = self.forward(X)
                loss = self._compute_loss(y, y_pred)
                print(f"Epoch {epoch:4d} | Loss: {loss:.6f}")
            elif epoch == epochs - 1:
                y_pred = self.forward(X)
                loss = self._compute_loss(y, y_pred)
                if verbose:
                    print(f"Epoch {epoch:4d} | Loss: {loss:.6f}")
        
        return self
    
    def _train_batch(self, X_batch, y_batch, learning_rate):
        y_pred = self.forward(X_batch)
        grad_output = self._loss_gradient(y_batch, y_pred)
        self.backward(grad_output, learning_rate)
    
    def _compute_loss(self, y_true, y_pred):
        return np.mean((y_true - y_pred) ** 2)

    def _loss_gradient(self, y_true, y_pred):
        return 2 * (y_pred - y_true) / y_true.shape[0]
    
    def predict(self, X):
        return self.forward(X)

    
    def score(self, X, y):
        if y.ndim == 1:
            y = y.reshape(-1, 1)
        y_pred = self.predict(X)
        
        return np.mean(np.abs(y - y_pred)), np.mean((y-y_pred)**2)

In [ ]:
data = pd.read_csv('ParisHousing.csv')
X = data.drop('price', axis=1).values
y = data['price'].values

scaler_X = StandardScaler()
X = scaler_X.fit_transform(X)


scaler_y = StandardScaler()
y = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Признаков: {X.shape[1]}, Объектов: {X.shape[0]}")


model = NeuralNetworRegration()

model.add(Dense(X.shape[1], 64, activation=ReLU()))
model.add(Dense(64, 128, activation=ReLU()))
model.add(Dense(128, 64, activation=ReLU()))
model.add(Dense(64, 32, activation=ReLU()))
model.add(Dense(32, 1, activation=None))

model.fit(X_train, y_train, epochs=200, learning_rate=0.01, batch_size=32, verbose=True)

y_pred = model.predict(X_test)

score = model.score(X_test, y_test)
print(f"\nMAE: {score[0]:.4f}, MSE: {score[1]:.4f}")



Тенсер Флоу для регрессии

In [ ]:


data = pd.read_csv('ParisHousing.csv')
X = data.drop('price', axis=1).values
y = data['price'].values

scaler_X = StandardScaler()
X = scaler_X.fit_transform(X)


scaler_y = StandardScaler()
y = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Признаков: {X.shape[1]}, Объектов: {X.shape[0]}")


model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(X.shape[1],)), 
    
    tf.keras.layers.Dense(64, activation='relu'), 
    tf.keras.layers.Dense(32, activation='relu'), 
    
    tf.keras.layers.Dense(1) 
])


model.compile(
    optimizer='adam',
    loss='mse',       
    metrics=['mae', 'mse']    
)

model.fit(X_train, y_train, epochs=20)
